# 08 — Migration Results

Consolidated view across every step: accounts, addresses, and orders.
Reads only from `migration_data/` — safe to re-run any time without
touching OneBill.

## 1. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

df_account_results  = try_load_df("account_results")
df_address_results  = try_load_df("address_results")
df_order_results    = try_load_df("order_results")


## 2. Account creation summary

In [ ]:
if df_account_results is not None:
    print(df_account_results["status"].value_counts().to_string())
    display(df_account_results)
else:
    print("04_Create_Accounts.ipynb has not been run yet.")


## 3. Address creation summary + failures

In [ ]:
if df_address_results is not None:
    print(df_address_results["status"].value_counts().to_string())
    address_failures = df_address_results[df_address_results["status"] != "created"]
    print(f"\n{len(address_failures):,} not created (failed or skipped)")
else:
    address_failures = pd.DataFrame()
    print("06_Create_Addresses.ipynb has not been run yet.")

address_failures.head(20)


In [ ]:
if not address_failures.empty:
    out_path = f'Address_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    address_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(address_failures):,} address failures/skips to {out_path}")


## 4. Order creation summary + failures

In [ ]:
if df_order_results is not None:
    print(df_order_results["status"].value_counts().to_string())
    order_failures = df_order_results[df_order_results["status"] == "failed"]
    print(f"\n{len(order_failures):,} failed orders")
else:
    order_failures = pd.DataFrame()
    print("07_Create_Subscription_Orders.ipynb has not been run yet.")

order_failures.head(20)


In [ ]:
if not order_failures.empty:
    out_path = f'Order_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    order_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(order_failures):,} order failures to {out_path}")

    error_summary = (
        order_failures.groupby("error")
        .agg(count=("SubscriptionUSN", "size"), example=("SubscriptionUSN", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
else:
    error_summary = pd.DataFrame()

error_summary


## 5. Orders on a fallback (unmatched) plan

Successfully created, but on `STATIC_FALLBACK_PLAN` rather than a matched
plan — worth reviewing before calling the migration done.

In [ ]:
if df_order_results is not None and not df_order_results.empty:
    unmatched_plans = df_order_results[
        (df_order_results["status"] == "success") & (df_order_results["plan_matched"] == False)  # noqa: E712
    ]
    print(f"{len(unmatched_plans):,} successfully-created orders used the STATIC_FALLBACK_PLAN")
else:
    unmatched_plans = pd.DataFrame()

unmatched_plans


## 6. End-to-end funnel

In [ ]:
_df_subs = try_load_df("subscriptions_resolved")
funnel = {
    "subscriptions loaded":     len(_df_subs) if _df_subs is not None else 0,
    "addresses created":        (df_address_results["status"] == "created").sum() if df_address_results is not None else 0,
    "orders attempted":         len(df_order_results) if df_order_results is not None else 0,
    "orders succeeded":         (df_order_results["status"] == "success").sum() if df_order_results is not None else 0,
}
pd.Series(funnel, name="count").to_frame()
